# Regression Data Preparation

This notebook prepares `player_season_regression_dataset_all3.csv` for modeling player market value.

Goals:
- profile null values
- distinguish true missing data from valid zeros/no-events
- add football-specific engineered features, including age
- build separate feature matrices by player position
- support two modeling scenarios: with previous-season market value and without it
- export processed datasets for regression and tree models

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# Resolve project root regardless of whether the notebook is run from root or /notebooks.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists()),
    Path.cwd(),
)

DATA_PATH = PROJECT_ROOT / "data/merged/player_season_regression_dataset_all3.csv"
HISTORICAL_OUTPUT_PATH = PROJECT_ROOT / "data/merged/player_value_modeling_2015_16_to_2024_25.csv"
PREDICTION_2025_OUTPUT_PATH = PROJECT_ROOT / "data/merged/player_stats_2025_26_null_market_value.csv"

assert DATA_PATH.exists(), f"Missing file: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)

# Stable row identifier for mapping predictions/regression residuals back to the original player-season row.
# The duplicate counter handles mid-season team rows or any repeated player-season keys.
uid_key_cols = ["api_player_id", "season", "api_team_id", "league_id"]
uid_key_frame = df[uid_key_cols].fillna("missing").astype(str)
df["player_season_duplicate_n"] = df.groupby(uid_key_cols, dropna=False).cumcount()
df["player_season_uid"] = (
    uid_key_frame["api_player_id"] + "_" +
    uid_key_frame["season"] + "_" +
    uid_key_frame["api_team_id"] + "_" +
    uid_key_frame["league_id"] + "_" +
    df["player_season_duplicate_n"].astype(str)
)

print(df.shape)
print("Unique player_season_uid:", df["player_season_uid"].is_unique)
df.head(3)

(24459, 93)
Unique player_season_uid: True


,api_player_id,api_player_name,api_team_id,api_team_name,api_league_id,api_season,api_position,api_squad_rows,api_appearances,api_starts,api_sub_appearances,api_minutes,api_avg_rating,api_offsides,api_shots_total,api_shots_on,api_goals_total,api_goals_conceded,api_goals_assists,api_goals_saves,api_passes_total,api_passes_key,api_tackles_total,api_tackles_blocks,api_tackles_interceptions,api_duels_total,api_duels_won,api_dribbles_attempts,api_dribbles_success,api_dribbles_past,api_fouls_drawn,api_fouls_committed,api_cards_yellow,api_cards_red,api_penalty_won,api_penalty_commited,api_penalty_scored,api_penalty_missed,api_penalty_saved,api_goals_total_per90,api_goals_assists_per90,api_shots_total_per90,api_shots_on_per90,api_passes_key_per90,api_tackles_total_per90,api_duels_won_per90,api_cards_yellow_per90,api_cards_red_per90,api_all_team_names_in_season,season,player_name,team_name,league_id,understat_league,api_height,api_weight,understat_season,understat_team,understat_player,understat_league_id,understat_season_id,understat_team_id,understat_player_id,understat_position,understat_matches,understat_minutes,understat_goals,understat_xg,understat_np_goals,understat_np_xg,understat_assists,understat_xa,understat_shots,understat_key_passes,understat_yellow_cards,understat_red_cards,understat_xg_chain,understat_xg_buildup,target_market_value_eur,market_value_date,transfermarkt_team_name,transfermarkt_age_at_season,transfermarkt_match_level,last_market_value_eur,last_market_value_date,last_transfermarkt_team_name,last_transfermarkt_match_level,matched_last_transfermarkt,understat_match_level,matched_understat,matched_transfermarkt,player_season_duplicate_n,player_season_uid
0,1706,Carlos Bacca,489,AC Milan,135,2015,F,38,38,38,0,3167,6.946,0,77,37,18,0,2,0,491,41,0,0,10,281,93,65,31,11,24,30,2,0,1,0,2,0,0,0.512,0.057,2.188,1.051,1.165,0.0,2.643,0.057,0.000,NaN,2015,Carlos Bacca,AC Milan,135,ITA-Serie A,181,77,1516,AC Milan,Carlos Bacca,2,2015,111,1125,F S,38,3179,18,14.870262,16,13.347665,2,3.657710,77,41,2,0,18.540677,3.277733,25000000.0,2016-07-15,AC Milan,28.0,team_match,25000000.0,2015-07-01,Sevilla FC,player_match,1,player_league,1,1,0,1706_2015_489_135_0
1,1632,Alessio Romagnoli,489,AC Milan,135,2015,D,34,34,34,0,2943,6.920,0,6,0,0,0,0,0,1306,5,0,0,84,252,146,8,5,10,22,46,10,1,0,2,0,0,0,0.000,0.000,0.183,0.000,0.153,0.0,4.465,0.306,0.031,NaN,2015,Alessio Romagnoli,AC Milan,135,ITA-Serie A,185,75,1516,AC Milan,Alessio Romagnoli,2,2015,111,1119,D S,34,2942,0,0.339300,0,0.339300,0,0.189771,6,5,8,1,4.090086,3.907433,18000000.0,2016-07-15,AC Milan,20.0,team_match,15000000.0,2015-09-23,AC Milan,team_match,1,player_league,1,1,0,1632_2015_489_135_0
2,1638,Giacomo Bonaventura,489,AC Milan,135,2015,M,33,33,31,2,2841,7.356,0,104,33,6,0,8,0,1191,70,0,0,40,414,204,118,76,23,82,43,8,0,1,0,0,0,0,0.190,0.253,3.295,1.045,2.218,0.0,6.463,0.253,0.000,NaN,2015,Giacomo Bonaventura,AC Milan,135,ITA-Serie A,180,75,1516,AC Milan,Giacomo Bonaventura,2,2015,111,1121,F M S,33,2845,6,7.727590,6,7.727590,8,7.649018,104,69,8,0,15.411559,6.036132,21000000.0,2016-07-15,AC Milan,25.0,team_match,15000000.0,2015-07-01,AC Milan,team_match,1,player_league,1,1,0,1638_2015_489_135_0


In [2]:
# Selective type cleanup (do NOT force all columns to numeric)
# Keep text/ID fields as-is, and only convert columns that are mostly numeric.
def parse_numeric_with_units(series):
    """Parse numeric strings, including measurements like '181 cm' or '77 kg'."""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    cleaned = (
        series.astype("string")
        .str.strip()
        .str.lower()
        .str.replace(",", "", regex=False)
        .str.replace(r"\s*(cm|kg|m|lbs?|eur|€)$", "", regex=True)
        .str.extract(r"([-+]?\d*\.?\d+)", expand=False)
    )
    return pd.to_numeric(cleaned, errors="coerce")


protected_text_tokens = [
    "name", "position", "league", "date", "match_level", "team_name"
]
protected_exact = {
    "player_name", "team_name", "understat_league", "transfermarkt_team_name",
    "transfermarkt_match_level", "understat_match_level", "api_position"
}

candidate_numeric = [
    c for c in df.columns
    if c not in protected_exact
    and not any(token in c.lower() for token in protected_text_tokens)
]

# Always treat target/season/measurements/value fields as numeric if present.
force_numeric = [
    c for c in [
        "target_market_value_eur", "last_market_value_eur", "season", "api_height", "api_weight",
        "transfermarkt_age_at_season"
    ]
    if c in df.columns
]

conversion_report = []
for col in candidate_numeric:
    raw = df[col]
    parsed = parse_numeric_with_units(raw)
    ratio = parsed.notna().mean()

    # Convert only when the column is mostly numeric, or explicitly required.
    if (col in force_numeric) or (ratio >= 0.95):
        df[col] = parsed
        conversion_report.append((col, round(float(ratio), 4), "converted"))
    else:
        conversion_report.append((col, round(float(ratio), 4), "kept_as_text"))

report_df = pd.DataFrame(conversion_report, columns=["column", "numeric_ratio", "action"])
print("Conversion summary:")
print(report_df["action"].value_counts())
print("\nColumns kept as text due to low numeric ratio:")
display(report_df[report_df["action"] == "kept_as_text"].sort_values("numeric_ratio").head(20))

# Log target for regression stability
df["target_market_value_log"] = np.log1p(df["target_market_value_eur"])
print(df[["target_market_value_eur", "target_market_value_log"]].describe().T)

Conversion summary:
action
converted       73
kept_as_text     2
Name: count, dtype: int64

Columns kept as text due to low numeric ratio:


,column,numeric_ratio,action
49,understat_player,0.0000,kept_as_text
48,understat_team,0.0292,kept_as_text


                           count          mean           std           min           25%           50%           75%           max
target_market_value_eur  22076.0  9.682552e+06  1.527403e+07  25000.000000  1.500000e+06  4.000000e+06  1.200000e+07  2.000000e+08
target_market_value_log  22076.0  1.517194e+01  1.438449e+00     10.126671  1.422098e+01  1.520181e+01  1.630042e+01  1.911383e+01


## Age, Height, And Weight Features

Age is a core market-value predictor in football valuation research. Papers such as Herm, Callsen-Bracker, and Kreis (2014) and Muller, Simons, and Weinmann (2017) combine player characteristics like age and position with performance, league, and team context.

This notebook now uses only columns already present in `player_season_regression_dataset_all3.csv`, especially `transfermarkt_age_at_season`, `api_height`, and `api_weight`. Age is kept continuous and non-linear terms are added because player value usually rises early, peaks in the mid-to-late 20s, and declines afterward.

In [3]:
# Use only fields already present in player_season_regression_dataset_all3.csv.
age_source_col = "transfermarkt_age_at_season" if "transfermarkt_age_at_season" in df.columns else None
if age_source_col is not None:
    df["player_age"] = pd.to_numeric(df[age_source_col], errors="coerce")
    df["player_age_sq"] = df["player_age"] ** 2
    df["age_peak_gap"] = df["player_age"] - 27
    df["is_u21"] = (df["player_age"] <= 21).astype("Int64")
    df["is_u23"] = (df["player_age"] <= 23).astype("Int64")
    df["is_over30"] = (df["player_age"] >= 30).astype("Int64")
else:
    print("WARNING: no age column found in the CSV. Expected transfermarkt_age_at_season.")

for source_col, feature_col in [("api_height", "player_height_cm"), ("api_weight", "player_weight_kg")]:
    if source_col in df.columns:
        df[feature_col] = parse_numeric_with_units(df[source_col])

if {"player_height_cm", "player_weight_kg"}.issubset(df.columns):
    height_m = df["player_height_cm"] / 100
    df["player_bmi"] = df["player_weight_kg"] / (height_m ** 2)

print("Age coverage:", round(df.get("player_age", pd.Series(dtype=float)).notna().mean() * 100, 2), "%")
display(df[[c for c in ["player_season_uid", "player_name", "season", "api_position", "player_age", "player_height_cm", "player_weight_kg", "player_bmi"] if c in df.columns]].head())

Age coverage: 99.1 %


,player_season_uid,player_name,season,api_position,player_age,player_height_cm,player_weight_kg,player_bmi
0,1706,Carlos Bacca,2015,F,28.0,181,77,23.503556
1,1632,Alessio Romagnoli,2015,D,20.0,185,75,21.913806
2,1638,Giacomo Bonaventura,2015,M,25.0,180,75,23.148148
3,1645,Riccardo Montolivo,2015,M,30.0,182,79,23.849777
4,1622,Gianluigi Donnarumma,2015,G,16.0,196,90,23.427738


In [4]:
# Null profile table
null_profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=True)
}).sort_values(["null_pct", "null_count"], ascending=False)

display(null_profile)

,dtype,null_count,null_pct,n_unique
api_all_team_names_in_season,object,23011,94.08,1133
last_market_value_eur,float64,2393,9.78,143
last_market_value_date,object,2393,9.78,511
last_transfermarkt_team_name,object,2393,9.78,1001
last_transfermarkt_match_level,object,2393,9.78,2
...,...,...,...,...
player_season_duplicate_n,int64,0,0.00,1
player_season_uid,Int64,0,0.00,7597
is_u21,Int64,0,0.00,2
is_u23,Int64,0,0.00,2


## Missing vs True Zero Rules

We treat some nulls as **valid zero/no-event** and others as **true missing**.

### Fill as zero (no event happened)
- `api_goals_total`, `api_goals_assists`, `api_shots_total`, `api_shots_on`, `api_passes_key`, `api_tackles_total`, `api_duels_won`, `api_cards_yellow`, `api_cards_red`
- corresponding `api_*_per90`
- `understat_goals`, `understat_assists`, `understat_shots`, `understat_key_passes`, `understat_yellow_cards`, `understat_red_cards`, `understat_xg`, `understat_np_goals`, `understat_np_xg`, `understat_xa`, `understat_xg_chain`, `understat_xg_buildup`

### Keep as missing (informationally unknown)
- ratings/position metadata (e.g. `api_avg_rating`, `api_position`)
- market metadata (`market_value_date`, `transfermarkt_team_name`) when absent

In [5]:
zero_fill_cols = [
    # API event-count columns
    "api_goals_total", "api_goals_assists", "api_shots_total", "api_shots_on",
    "api_passes_key", "api_tackles_total", "api_duels_won", "api_cards_yellow", "api_cards_red",
    "api_goals_total_per90", "api_goals_assists_per90", "api_shots_total_per90", "api_shots_on_per90",
    "api_passes_key_per90", "api_tackles_total_per90", "api_duels_won_per90",
    "api_offsides", "api_goals_conceded", "api_goals_saves", "api_passes_total",
    "api_tackles_blocks", "api_tackles_interceptions", "api_duels_total",
    "api_dribbles_attempts", "api_dribbles_success", "api_dribbles_past",
    "api_fouls_drawn", "api_fouls_committed", "api_penalty_won", "api_penalty_commited",
    "api_penalty_scored", "api_penalty_missed", "api_penalty_saved",
    # Understat event-count and expected-event columns
    "understat_goals", "understat_assists", "understat_shots", "understat_key_passes",
    "understat_yellow_cards", "understat_red_cards", "understat_xg", "understat_np_goals",
    "understat_np_xg", "understat_xa", "understat_xg_chain", "understat_xg_buildup",
]

existing_zero_fill_cols = [c for c in zero_fill_cols if c in df.columns]
for col in existing_zero_fill_cols:
    df[col] = df[col].fillna(0)

# Common numeric imputations where null usually means unknown rather than zero.
if "api_avg_rating" in df.columns:
    df["api_avg_rating_missing"] = df["api_avg_rating"].isna().astype(int)
    df["api_avg_rating"] = df["api_avg_rating"].fillna(df["api_avg_rating"].median())

if "api_position" in df.columns:
    df["api_position"] = df["api_position"].fillna("Unknown")

# Keep position coarse and explicit so models are trained separately by role.
position_map = {
    "Goalkeeper": "G", "Defender": "D", "Midfielder": "M", "Attacker": "F",
    "G": "G", "D": "D", "M": "M", "F": "F", "SUB": "SUB", "Unknown": "Unknown",
}
df["model_position"] = df.get("api_position", "Unknown").map(position_map).fillna(df.get("api_position", "Unknown"))

rows_before_sub_drop = len(df)
df = df[df["model_position"] != "SUB"].copy()
print(f"Dropped SUB rows: {rows_before_sub_drop - len(df):,}")

# Requested null policy:
# - drop rows when a non-target column has a null rate below 10%
# - notify when any non-target column has a null rate above 10%
# Columns with higher null rates are retained here so the modeling code can decide whether to impute or exclude them.
target_nullable_cols = {
    "target_market_value_eur", "target_market_value_log", "market_value_date",
    "last_market_value_eur", "last_market_value_log", "last_market_value_date",
    "last_transfermarkt_team_name", "last_transfermarkt_match_level", "matched_last_transfermarkt",
}
null_drop_threshold = 0.10
null_rate = df.isna().mean().sort_values(ascending=False)
null_rate_for_drop = null_rate.drop(labels=[c for c in target_nullable_cols if c in null_rate.index])
low_null_cols = null_rate_for_drop[(null_rate_for_drop > 0) & (null_rate_for_drop < null_drop_threshold)].index.tolist()
high_null_report = (
    pd.DataFrame({
        "column": null_rate_for_drop[null_rate_for_drop > null_drop_threshold].index,
        "null_rate_pct": (null_rate_for_drop[null_rate_for_drop > null_drop_threshold].values * 100).round(2),
        "null_count": df[null_rate_for_drop[null_rate_for_drop > null_drop_threshold].index].isna().sum().values,
    })
    .sort_values("null_rate_pct", ascending=False)
)

rows_before_null_drop = len(df)
if low_null_cols:
    df = df.dropna(subset=low_null_cols).copy()

print(f"Rows before low-null drop: {rows_before_null_drop:,}")
print(f"Rows after low-null drop: {len(df):,}")
print(f"Dropped rows: {rows_before_null_drop - len(df):,}")
print("Columns used for low-null row drop (<10% null, excluding target fields):", low_null_cols)

if not high_null_report.empty:
    print("WARNING: columns with >10% null rate retained for review:")
    display(high_null_report)
else:
    print("No non-target columns have >10% null rate.")

# Keep all retained rows, including 2025-26 rows whose market value target is intentionally null.
df_model = df.copy()

if "last_market_value_eur" not in df_model.columns:
    raise ValueError("source CSV has no last_market_value_eur column; cannot enforce last-market-value filter.")

df_model["last_market_value_eur"] = pd.to_numeric(df_model["last_market_value_eur"], errors="coerce")
rows_before_last_value_filter = len(df_model)
df_model = df_model[df_model["last_market_value_eur"].notna()].copy()
print(f"Dropped rows with no last market value: {rows_before_last_value_filter - len(df_model):,}")

print("Rows retained for export:", len(df_model))
print("Rows with target:", df_model["target_market_value_eur"].notna().sum())
print("Rows with null target:", df_model["target_market_value_eur"].isna().sum())
print("Rows by model position:")
print(df_model["model_position"].value_counts(dropna=False).sort_index())
print("\nRemaining nulls in retained table:")
(df_model.isna().mean().sort_values(ascending=False).head(20) * 100).round(2)

Dropped SUB rows: 4
Rows before low-null drop: 24,455
Rows after low-null drop: 23,841
Dropped rows: 614
Columns used for low-null row drop (<10% null, excluding target fields): ['player_bmi', 'player_weight_kg', 'api_weight', 'api_height', 'player_height_cm', 'player_age', 'transfermarkt_age_at_season', 'player_age_sq', 'age_peak_gap', 'transfermarkt_team_name', 'transfermarkt_match_level', 'api_cards_yellow_per90', 'api_cards_red_per90']


,column,null_rate_pct,null_count
0,api_all_team_names_in_season,94.08,23007


Dropped rows with no last market value: 2,066
Rows retained for export: 21775
Rows with target: 19718
Rows with null target: 2057
Rows by model position:
model_position
D    7162
F    4703
G    1773
M    8137
Name: count, dtype: int64

Remaining nulls in retained table:


api_all_team_names_in_season    93.75
target_market_value_eur          9.45
target_market_value_log          9.45
api_player_name                  0.00
api_league_id                    0.00
api_season                       0.00
api_team_id                      0.00
api_team_name                    0.00
api_player_id                    0.00
api_starts                       0.00
api_sub_appearances              0.00
api_minutes                      0.00
api_avg_rating                   0.00
api_offsides                     0.00
api_shots_total                  0.00
api_shots_on                     0.00
api_goals_total                  0.00
api_goals_conceded               0.00
api_goals_assists                0.00
api_goals_saves                  0.00
dtype: float64

In [6]:
def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan) if isinstance(denominator, pd.Series) else denominator
    return numerator / denominator


# Exposure and playing-time features.
if "api_minutes" in df_model.columns:
    df_model["log_minutes"] = np.log1p(df_model["api_minutes"])
    df_model["minutes_share_3420"] = (df_model["api_minutes"] / (38 * 90)).clip(0, 1)
    if "api_appearances" in df_model.columns:
        df_model["minutes_per_appearance"] = safe_divide(df_model["api_minutes"], df_model["api_appearances"]).fillna(0)
    if "api_starts" in df_model.columns:
        df_model["start_rate"] = safe_divide(df_model["api_starts"], df_model.get("api_appearances", pd.Series(np.nan, index=df_model.index))).fillna(0)
        df_model["minutes_per_start"] = safe_divide(df_model["api_minutes"], df_model["api_starts"]).fillna(0)
    if "api_sub_appearances" in df_model.columns:
        df_model["sub_appearance_rate"] = safe_divide(df_model["api_sub_appearances"], df_model.get("api_appearances", pd.Series(np.nan, index=df_model.index))).fillna(0)

# Understat per-90 features commonly used for attacking contribution and possession value.
understat_minutes = df_model.get("understat_minutes", df_model.get("api_minutes", pd.Series(np.nan, index=df_model.index)))
for col in [
    "understat_goals", "understat_xg", "understat_np_goals", "understat_np_xg", "understat_assists",
    "understat_xa", "understat_shots", "understat_key_passes", "understat_xg_chain", "understat_xg_buildup"
]:
    if col in df_model.columns:
        df_model[f"{col}_per90"] = safe_divide(df_model[col] * 90, understat_minutes).fillna(0)

# Efficiency and style features. These are useful for both linear models and trees.
if {"api_shots_on", "api_shots_total"}.issubset(df_model.columns):
    df_model["shot_on_target_rate"] = safe_divide(df_model["api_shots_on"], df_model["api_shots_total"]).fillna(0)
if {"api_goals_total", "api_shots_total"}.issubset(df_model.columns):
    df_model["goal_conversion_rate"] = safe_divide(df_model["api_goals_total"], df_model["api_shots_total"]).fillna(0)
if {"api_dribbles_success", "api_dribbles_attempts"}.issubset(df_model.columns):
    df_model["dribble_success_rate"] = safe_divide(df_model["api_dribbles_success"], df_model["api_dribbles_attempts"]).fillna(0)
if {"api_duels_won", "api_duels_total"}.issubset(df_model.columns):
    df_model["duel_win_rate"] = safe_divide(df_model["api_duels_won"], df_model["api_duels_total"]).fillna(0)
if {"understat_goals", "understat_xg"}.issubset(df_model.columns):
    df_model["goals_minus_xg"] = df_model["understat_goals"] - df_model["understat_xg"]
if {"understat_np_goals", "understat_np_xg"}.issubset(df_model.columns):
    df_model["np_goals_minus_np_xg"] = df_model["understat_np_goals"] - df_model["understat_np_xg"]
if {"understat_assists", "understat_xa"}.issubset(df_model.columns):
    df_model["assists_minus_xa"] = df_model["understat_assists"] - df_model["understat_xa"]
if {"understat_xg_per90", "understat_xa_per90"}.issubset(df_model.columns):
    df_model["xg_xa_per90"] = df_model["understat_xg_per90"] + df_model["understat_xa_per90"]
if {"understat_np_xg_per90", "understat_xa_per90"}.issubset(df_model.columns):
    df_model["np_xg_xa_per90"] = df_model["understat_np_xg_per90"] + df_model["understat_xa_per90"]
if {"api_goals_total_per90", "api_goals_assists_per90"}.issubset(df_model.columns):
    df_model["goals_assists_per90"] = df_model["api_goals_total_per90"] + df_model["api_goals_assists_per90"]
if {"api_cards_yellow", "api_cards_red", "api_minutes"}.issubset(df_model.columns):
    df_model["cards_per90"] = safe_divide((df_model["api_cards_yellow"] + 3 * df_model["api_cards_red"]) * 90, df_model["api_minutes"]).fillna(0)

# Within-position season percentiles help compare a player to realistic peers.
percentile_cols = [
    "api_minutes", "api_avg_rating", "goals_assists_per90", "xg_xa_per90", "np_xg_xa_per90",
    "understat_xg_chain_per90", "understat_xg_buildup_per90", "api_tackles_total_per90",
    "api_duels_won_per90", "api_passes_key_per90", "shot_on_target_rate", "duel_win_rate",
]
for col in [c for c in percentile_cols if c in df_model.columns]:
    df_model[f"{col}_pos_season_pct"] = (
        df_model.groupby(["season", "model_position"])[col]
        .rank(pct=True, method="average")
        .fillna(0.5)
    )

# Use the last market value already included in the source CSV.
# Do not recompute it here, so the notebook keeps the dataset builder as the single source of truth.
df_model["last_market_value_log"] = np.log1p(df_model["last_market_value_eur"])
df_model["has_last_market_value"] = 1

# Keep the target in log space because market value is heavily right-skewed.
target_cols = ["target_market_value_eur", "target_market_value_log"]
leakage_cols = {
    "market_value_date", "transfermarkt_team_name", "transfermarkt_match_level", "understat_match_level",
    "last_market_value_date", "last_transfermarkt_team_name", "last_transfermarkt_match_level",
    "matched_understat", "matched_transfermarkt", "matched_last_transfermarkt", "date_of_birth", "player_join_name",
}
id_or_text_cols = {
    "player_season_uid", "player_season_duplicate_n",
    "api_player_id", "api_player_name", "api_team_id", "api_team_name", "player_name", "team_name",
    "understat_player", "understat_team", "understat_player_id", "understat_team_id", "understat_season_id",
    "api_season", "api_position", "understat_position", "model_position", "api_nationality",
}
scenario_specific_cols = {"last_market_value_eur", "last_market_value_log"}

numeric_feature_cols = [
    c for c in df_model.select_dtypes(include=[np.number, "bool"]).columns
    if c not in set(target_cols) | leakage_cols | id_or_text_cols
    and not c.endswith("_id")
]
context_categorical_cols = [c for c in ["league_id", "understat_league"] if c in df_model.columns]
base_feature_cols = numeric_feature_cols + context_categorical_cols

features_without_last_value = [c for c in base_feature_cols if c not in scenario_specific_cols]
features_with_last_value = base_feature_cols

def make_feature_matrix(input_df, feature_cols):
    categorical_cols = [c for c in context_categorical_cols if c in feature_cols]
    X = pd.get_dummies(input_df[feature_cols], columns=categorical_cols, dummy_na=True, drop_first=False)
    X = X.replace([np.inf, -np.inf], np.nan)
    return X.fillna(X.median(numeric_only=True)).fillna(0)

position_model_matrices = {}
position_summary = []
y_col = "target_market_value_log"

for position, position_df in df_model.groupby("model_position", dropna=False):
    position_key = str(position)
    X_without_last = make_feature_matrix(position_df, features_without_last_value)
    X_with_last = make_feature_matrix(position_df, features_with_last_value)
    y_position = position_df[y_col].copy()

    metadata_cols = [
        c for c in [
            "player_season_uid", "player_name", "season", "team_name", "league_id",
            "api_player_id", "api_team_id", "model_position"
        ]
        if c in position_df.columns
    ]
    position_model_matrices[position_key] = {
        "without_last_value": {"X": X_without_last, "y": y_position, "rows": position_df.index, "metadata": position_df[metadata_cols].copy()},
        "with_last_value": {"X": X_with_last, "y": y_position, "rows": position_df.index, "metadata": position_df[metadata_cols].copy()},
    }
    position_summary.append({
        "model_position": position_key,
        "rows": len(position_df),
        "features_without_last_value": X_without_last.shape[1],
        "features_with_last_value": X_with_last.shape[1],
        "last_value_coverage_pct": round(position_df["has_last_market_value"].mean() * 100, 2),
    })

position_feature_summary = pd.DataFrame(position_summary).sort_values("model_position")
display(position_feature_summary)

# Example matrix for one position/scenario.
example_position = position_feature_summary.iloc[0]["model_position"]
print(f"Example position: {example_position}")
position_model_matrices[example_position]["without_last_value"]["X"].head()

,model_position,rows,features_without_last_value,features_with_last_value,last_value_coverage_pct
0,D,7162,122,124,100.0
1,F,4703,122,124,100.0
2,G,1773,122,124,100.0
3,M,8137,122,124,100.0


Example position: D


,api_squad_rows,api_appearances,api_starts,api_sub_appearances,api_minutes,api_avg_rating,api_offsides,api_shots_total,api_shots_on,api_goals_total,api_goals_conceded,api_goals_assists,api_goals_saves,api_passes_total,api_passes_key,api_tackles_total,api_tackles_blocks,api_tackles_interceptions,api_duels_total,api_duels_won,api_dribbles_attempts,api_dribbles_success,api_dribbles_past,api_fouls_drawn,api_fouls_committed,api_cards_yellow,api_cards_red,api_penalty_won,api_penalty_commited,api_penalty_scored,api_penalty_missed,api_penalty_saved,api_goals_total_per90,api_goals_assists_per90,api_shots_total_per90,api_shots_on_per90,api_passes_key_per90,api_tackles_total_per90,api_duels_won_per90,api_cards_yellow_per90,api_cards_red_per90,season,api_height,api_weight,understat_season,understat_matches,understat_minutes,understat_goals,understat_xg,understat_np_goals,understat_np_xg,understat_assists,understat_xa,understat_shots,understat_key_passes,understat_yellow_cards,understat_red_cards,understat_xg_chain,understat_xg_buildup,transfermarkt_age_at_season,player_age,player_age_sq,age_peak_gap,is_u21,is_u23,is_over30,player_height_cm,player_weight_kg,player_bmi,api_avg_rating_missing,log_minutes,minutes_share_3420,minutes_per_appearance,start_rate,minutes_per_start,sub_appearance_rate,understat_goals_per90,understat_xg_per90,understat_np_goals_per90,understat_np_xg_per90,understat_assists_per90,understat_xa_per90,understat_shots_per90,understat_key_passes_per90,understat_xg_chain_per90,understat_xg_buildup_per90,shot_on_target_rate,goal_conversion_rate,dribble_success_rate,duel_win_rate,goals_minus_xg,np_goals_minus_np_xg,assists_minus_xa,xg_xa_per90,np_xg_xa_per90,goals_assists_per90,cards_per90,api_minutes_pos_season_pct,api_avg_rating_pos_season_pct,goals_assists_per90_pos_season_pct,xg_xa_per90_pos_season_pct,np_xg_xa_per90_pos_season_pct,understat_xg_chain_per90_pos_season_pct,understat_xg_buildup_per90_pos_season_pct,api_tackles_total_per90_pos_season_pct,api_duels_won_per90_pos_season_pct,api_passes_key_per90_pos_season_pct,shot_on_target_rate_pos_season_pct,duel_win_rate_pos_season_pct,has_last_market_value,league_id_39.0,league_id_61.0,league_id_78.0,league_id_135.0,league_id_140.0,league_id_nan,understat_league_ENG-Premier League,understat_league_ESP-La Liga,understat_league_FRA-Ligue 1,understat_league_GER-Bundesliga,understat_league_ITA-Serie A,understat_league_nan
1,34,34,34,0,2943,6.920,0,6,0,0,0,0,0,1306,5,0,0,84,252,146,8,5,10,22,46,10,1,0,2,0,0,0,0.000,0.000,0.183,0.000,0.153,0.0,4.465,0.306,0.031,2015,185,75,1516,34,2942,0,0.339300,0,0.339300,0,0.189771,6,5,8,1,4.090086,3.907433,20.0,20.0,400.0,-7.0,1,1,0,185,75,21.913806,0,7.987524,0.860526,86.558824,1.000000,86.558824,0.000000,0.000000,0.010380,0.000000,0.010380,0.000000,0.005805,0.183549,0.152957,0.125122,0.119534,0.000000,0.000000,0.625000,0.579365,-0.339300,-0.339300,-0.189771,0.016185,0.016185,0.000,0.397554,0.918571,0.547857,0.183571,0.107143,0.107143,0.288571,0.357143,0.377857,0.312857,0.249286,0.122857,0.514286,1,False,False,False,True,False,False,False,False,False,False,True,False
5,28,28,27,1,2334,7.206,0,19,5,3,0,1,0,967,17,0,0,57,260,184,33,20,16,61,16,2,0,0,0,0,0,0,0.116,0.039,0.733,0.193,0.656,0.0,7.095,0.077,0.000,2015,184,79,1516,28,2333,3,2.096465,3,2.096465,1,1.589682,19,17,2,0,5.134562,4.295593,28.0,28.0,784.0,1.0,0,0,0,184,79,23.334121,0,7.755767,0.682456,83.357143,0.964286,86.444444,0.035714,0.115731,0.080875,0.115731,0.080875,0.038577,0.061325,0.732962,0.655808,0.198076,0.165711,0.263158,0.157895,0.606061,0.707692,0.903535,0.903535,-0.589682,0.142200,0.142200,0.155,0.077121,0.734286,0.932143,0.817857,0.841429,0.847143,0.547143,0.580000,0.377857,0.875714,0.740000,0.516429,0.965714,1,False,False,False,True,False,False,False,False,False,False,True,False
6,27,27,27,0,2254,7.235,0,4,2,1,0,1,0,994,18,0,0,76,216,133,36,25,13,16,37,7,0,0,0,0,0,0,0.040,0.040,0.160,0.080,0.719,0.0,5.311,0.280,0.000,2015,180,73,1516,27,2254,1,0.192637,1,0.192637,1,2.003

In [7]:
# Save only the two requested files.
# 1) Historical training/modeling table: 2015-16 through 2024-25, with target market value and last market value when available.
# 2) 2025-26 player stats table: same engineered columns, but market value target intentionally set to null for prediction.
historical_model = df_model[
    df_model["season"].between(2015, 2024)
    & df_model["target_market_value_eur"].notna()
].copy()

prediction_2025 = df_model[df_model["season"].eq(2025)].copy()
for col in ["target_market_value_eur", "target_market_value_log", "market_value_date"]:
    if col in prediction_2025.columns:
        prediction_2025[col] = pd.NA

# Keep rows easy to map back to players, teams, positions, and seasons.
front_cols = [
    c for c in [
        "player_season_uid", "player_name", "season", "team_name", "league_id",
        "api_player_id", "api_team_id", "model_position", "api_position",
        "target_market_value_eur", "target_market_value_log",
        "last_market_value_eur", "last_market_value_log", "has_last_market_value"
    ]
    if c in df_model.columns
]
remaining_cols = [c for c in df_model.columns if c not in front_cols]
export_cols = front_cols + remaining_cols

historical_model[export_cols].to_csv(HISTORICAL_OUTPUT_PATH, index=False)
prediction_2025[export_cols].to_csv(PREDICTION_2025_OUTPUT_PATH, index=False)

print(f"Saved historical 2015-16 to 2024-25 file: {HISTORICAL_OUTPUT_PATH} | rows={len(historical_model):,} cols={len(export_cols):,}")
print(f"Saved 2025-26 null-target file: {PREDICTION_2025_OUTPUT_PATH} | rows={len(prediction_2025):,} cols={len(export_cols):,}")
print("Historical last market value coverage:", round(historical_model["has_last_market_value"].mean() * 100, 2), "%")
print("2025-26 last market value coverage:", round(prediction_2025["has_last_market_value"].mean() * 100, 2), "%" if len(prediction_2025) else "")

Saved historical 2015-16 to 2024-25 file: /Users/james/Desktop/sports_analytics_final_project/data/merged/player_value_modeling_2015_16_to_2024_25.csv | rows=19,718 cols=146
Saved 2025-26 null-target file: /Users/james/Desktop/sports_analytics_final_project/data/merged/player_stats_2025_26_null_market_value.csv | rows=2,057 cols=146
Historical last market value coverage: 100.0 %
2025-26 last market value coverage: 100.0 %


## Data Report: Dictionary, Distributions, And Sources

This section creates a reusable data report from the exported tables:

- `data/merged/player_value_modeling_2015_16_to_2024_25.csv` (historical/training)
- `data/merged/player_stats_2025_26_null_market_value.csv` (prediction set)

It outputs:
- a column-level data dictionary with inferred meaning and source
- numeric and categorical distribution summaries
- a markdown report for quick review and sharing

In [8]:
REPORT_DIR = Path("data/reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

report_hist = pd.read_csv(HISTORICAL_OUTPUT_PATH)
report_pred = pd.read_csv(PREDICTION_2025_OUTPUT_PATH)


def infer_source(col):
    if col.startswith("api_"):
        return "API-Football"
    if col.startswith("understat_"):
        return "Understat"
    if col.startswith("transfermarkt_") or col.startswith("last_transfermarkt_"):
        return "Transfermarkt"
    if col.startswith("target_") or col.startswith("last_market_") or col == "market_value_date":
        return "Transfermarkt target layer"
    if col in {"player_season_uid", "player_season_duplicate_n", "model_position"}:
        return "Engineered key/label"
    if col.endswith("_missing") or col.endswith("_pct") or col.startswith("is_"):
        return "Engineered feature"
    if col in {"season", "player_name", "team_name", "league_id", "understat_league"}:
        return "Merged identifier/context"
    return "Engineered/Merged"


def infer_meaning(col):
    custom = {
        "player_season_uid": "Stable unique identifier for player-season-team row",
        "model_position": "Coarse position group used for model segmentation (G, D, M, F)",
        "target_market_value_eur": "Current season market value target in EUR",
        "target_market_value_log": "Log-transformed target market value: log1p(target_market_value_eur)",
        "last_market_value_eur": "Previous market value from source data in EUR",
        "last_market_value_log": "Log-transformed previous market value: log1p(last_market_value_eur)",
        "has_last_market_value": "Flag (1/0) indicating previous market value availability",
        "transfermarkt_age_at_season": "Player age aligned to season from Transfermarkt",
        "player_age": "Model age feature (from transfermarkt_age_at_season)",
        "player_height_cm": "Player height in centimeters",
        "player_weight_kg": "Player weight in kilograms",
        "player_bmi": "Body Mass Index computed from height and weight",
    }
    if col in custom:
        return custom[col]
    if col.endswith("_per90"):
        return "Rate per 90 minutes"
    if col.endswith("_missing"):
        return "Missing-value indicator flag for companion column"
    if col.endswith("_pos_season_pct"):
        return "Percentile rank within season and model position"
    if col.startswith("api_"):
        return "API-Football player-season stat/attribute"
    if col.startswith("understat_"):
        return "Understat player-season stat/attribute"
    if col.startswith("transfermarkt_"):
        return "Transfermarkt attribute"
    if col.startswith("last_transfermarkt_"):
        return "Transfermarkt metadata for previous market value"
    return "Merged or engineered feature"


def top_values(series, n=5):
    vc = series.value_counts(dropna=False).head(n)
    return "; ".join([f"{str(idx)} ({int(val)})" for idx, val in vc.items()])


def build_column_dictionary(df, dataset_name):
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({
            "dataset": dataset_name,
            "column": col,
            "dtype": str(s.dtype),
            "null_pct": round(float(s.isna().mean() * 100), 2),
            "n_unique": int(s.nunique(dropna=True)),
            "source": infer_source(col),
            "meaning": infer_meaning(col),
            "example_values": top_values(s, n=3),
        })
    return pd.DataFrame(rows)


def numeric_distribution(df, dataset_name):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not num_cols:
        return pd.DataFrame()
    desc = df[num_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T.reset_index()
    desc = desc.rename(columns={"index": "column"})
    desc.insert(0, "dataset", dataset_name)
    desc["skew"] = df[num_cols].skew(numeric_only=True).values
    return desc


def categorical_distribution(df, dataset_name):
    cat_cols = [c for c in df.columns if c not in df.select_dtypes(include=[np.number]).columns]
    rows = []
    for col in cat_cols:
        s = df[col]
        rows.append({
            "dataset": dataset_name,
            "column": col,
            "null_pct": round(float(s.isna().mean() * 100), 2),
            "n_unique": int(s.nunique(dropna=True)),
            "top_values": top_values(s, n=8),
        })
    return pd.DataFrame(rows)


column_dictionary = pd.concat([
    build_column_dictionary(report_hist, "historical_2015_2024"),
    build_column_dictionary(report_pred, "prediction_2025"),
], ignore_index=True)

numeric_summary = pd.concat([
    numeric_distribution(report_hist, "historical_2015_2024"),
    numeric_distribution(report_pred, "prediction_2025"),
], ignore_index=True)

categorical_summary = pd.concat([
    categorical_distribution(report_hist, "historical_2015_2024"),
    categorical_distribution(report_pred, "prediction_2025"),
], ignore_index=True)

column_dictionary_path = REPORT_DIR / "player_value_data_dictionary.csv"
numeric_summary_path = REPORT_DIR / "player_value_numeric_distribution.csv"
categorical_summary_path = REPORT_DIR / "player_value_categorical_distribution.csv"

column_dictionary.to_csv(column_dictionary_path, index=False)
numeric_summary.to_csv(numeric_summary_path, index=False)
categorical_summary.to_csv(categorical_summary_path, index=False)

source_summary = pd.DataFrame(
    column_dictionary.groupby(["dataset", "source"], as_index=False)["column"].count()
).rename(columns={"column": "column_count"})
source_summary_path = REPORT_DIR / "player_value_source_summary.csv"
source_summary.to_csv(source_summary_path, index=False)

md_lines = []
md_lines.append("# Player Value Data Report")
md_lines.append("")
md_lines.append("## Data Sources")
md_lines.append("- API-Football: player season and per-90 football stats (`api_*`)")
md_lines.append("- Understat: advanced xG/xA and possession-chain features (`understat_*`)")
md_lines.append("- Transfermarkt: market value labels and age (`target_*`, `last_market_*`, `transfermarkt_*`)")
md_lines.append("- Engineered: keys and transformed features (e.g., `player_season_uid`, percentile features)")
md_lines.append("")
md_lines.append("## Dataset Shapes")
md_lines.append(f"- Historical (2015-16 to 2024-25): {report_hist.shape[0]:,} rows x {report_hist.shape[1]:,} columns")
md_lines.append(f"- Prediction (2025-26): {report_pred.shape[0]:,} rows x {report_pred.shape[1]:,} columns")
md_lines.append("")
md_lines.append("## Key Completeness Checks")
for name, df_check in [("historical", report_hist), ("prediction_2025", report_pred)]:
    uid_null = round(float(df_check["player_season_uid"].isna().mean() * 100), 2) if "player_season_uid" in df_check.columns else None
    last_null = round(float(df_check["last_market_value_eur"].isna().mean() * 100), 2) if "last_market_value_eur" in df_check.columns else None
    target_null = round(float(df_check["target_market_value_eur"].isna().mean() * 100), 2) if "target_market_value_eur" in df_check.columns else None
    md_lines.append(f"- {name}: uid null%={uid_null}, last_market_value null%={last_null}, target null%={target_null}")

md_lines.append("")
md_lines.append("## Output Files")
md_lines.append(f"- Data dictionary: `{column_dictionary_path}`")
md_lines.append(f"- Numeric distributions: `{numeric_summary_path}`")
md_lines.append(f"- Categorical distributions: `{categorical_summary_path}`")
md_lines.append(f"- Source summary: `{source_summary_path}`")

report_md_path = REPORT_DIR / "player_value_data_report.md"
report_md_path.write_text("\n".join(md_lines), encoding="utf-8")

print(f"Saved data dictionary: {column_dictionary_path}")
print(f"Saved numeric distributions: {numeric_summary_path}")
print(f"Saved categorical distributions: {categorical_summary_path}")
print(f"Saved source summary: {source_summary_path}")
print(f"Saved markdown report: {report_md_path}")

print("\nSource counts by dataset:")
display(source_summary.sort_values(["dataset", "column_count"], ascending=[True, False]))

print("\nColumn dictionary preview:")
display(column_dictionary)

Saved data dictionary: data/reports/player_value_data_dictionary.csv
Saved numeric distributions: data/reports/player_value_numeric_distribution.csv
Saved categorical distributions: data/reports/player_value_categorical_distribution.csv
Saved source summary: data/reports/player_value_source_summary.csv
Saved markdown report: data/reports/player_value_data_report.md

Source counts by dataset:


,dataset,source,column_count
0,historical_2015_2024,API-Football,57
7,historical_2015_2024,Understat,36
3,historical_2015_2024,Engineered/Merged,27
1,historical_2015_2024,Engineered feature,8
6,historical_2015_2024,Transfermarkt target layer,6
5,historical_2015_2024,Transfermarkt,5
4,historical_2015_2024,Merged identifier/context,4
2,historical_2015_2024,Engineered key/label,3
8,prediction_2025,API-Football,57
15,prediction_2025,Understat,36



Column dictionary preview:


,dataset,column,dtype,null_pct,n_unique,source,meaning,example_values
0,historical_2015_2024,player_season_uid,int64,0.0,6360,Engineered key/label,Stable unique identifier for player-season-tea...,203 (10); 30533 (10); 2807 (10)
1,historical_2015_2024,player_name,object,0.0,6412,Merged identifier/context,Merged or engineered feature,Danilo (23); Adama Traoré (14); Marcelo (13)
2,historical_2015_2024,season,int64,0.0,10,Merged identifier/context,Merged or engineered feature,2022 (2313); 2021 (2308); 2023 (2254)
3,historical_2015_2024,team_name,object,0.0,168,Merged identifier/context,Merged or engineered feature,Bologna (264); Fiorentina (258); Juventus (257)
4,historical_2015_2024,league_id,int64,0.0,5,Merged identifier/context,Merged or engineered feature,135 (4887); 78 (3864); 39 (3726)
...,...,...,...,...,...,...,...,...
287,prediction_2025,api_tackles_total_per90_pos_season_pct,float64,0.0,1575,API-Football,Percentile rank within season and model position,0.34 (101); 0.0450819672131147 (43); 0.0210144...
288,prediction_2025,api_duels_won_per90_pos_season_pct,float64,0.0,1817,API-Football,Percentile rank within season and model position,0.0533333333333333 (15); 0.0122950819672131 (1...
289,prediction_2025,api_passes_key_per90_pos_season_pct,float64,0.0,1497,API-Football,Percentile rank within season and model position,0.31 (92); 0.0623188405797101 (85); 0.04098360...
290,prediction_2025,shot_on_target_rate_pos_season_pct,float64,0.0,399,Engineered feature,Percentile rank within season and model position,0.1413043478260869 (194); 0.5 (149); 0.0624142...


In [9]:
#print all columns
for col in report_hist.columns:
    print(col)
    


player_season_uid
player_name
season
team_name
league_id
api_player_id
api_team_id
model_position
api_position
target_market_value_eur
target_market_value_log
last_market_value_eur
last_market_value_log
has_last_market_value
api_player_name
api_team_name
api_league_id
api_season
api_squad_rows
api_appearances
api_starts
api_sub_appearances
api_minutes
api_avg_rating
api_offsides
api_shots_total
api_shots_on
api_goals_total
api_goals_conceded
api_goals_assists
api_goals_saves
api_passes_total
api_passes_key
api_tackles_total
api_tackles_blocks
api_tackles_interceptions
api_duels_total
api_duels_won
api_dribbles_attempts
api_dribbles_success
api_dribbles_past
api_fouls_drawn
api_fouls_committed
api_cards_yellow
api_cards_red
api_penalty_won
api_penalty_commited
api_penalty_scored
api_penalty_missed
api_penalty_saved
api_goals_total_per90
api_goals_assists_per90
api_shots_total_per90
api_shots_on_per90
api_passes_key_per90
api_tackles_total_per90
api_duels_won_per90
api_cards_yellow_per90